# Setup and the course LLM client

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

Run this notebook first. It checks your environment, seeds the replay cache from the recorded responses in `data/llm_fixtures/`, and shows the client from the orientation slides (parts 1 to 4) at work. Replay mode needs no API key.

In [ ]:
import importlib
for pkg in ["yaml", "numpy", "pandas", "sklearn", "openpyxl", "bs4", "pydantic", "duckdb", "chromadb", "rank_bm25", "langchain_text_splitters", "fastapi", "matplotlib"]:
    try:
        importlib.import_module(pkg); print(f"ok       {pkg}")
    except ImportError:
        print(f"MISSING  {pkg}  ->  pip install -r requirements.txt")

In [ ]:
import subprocess
subprocess.run([sys.executable, "tools/seed_replay_cache.py"], check=True)

`src/llm/client.py` is the slides' code (parts 1, 3, 2, 4 in dependency order) plus a small adapter for live calls. Every model and embedding call in Case Study 2 goes through it.

In [ ]:
import inspect
print(inspect.getsource(llm.structured))

In [ ]:
# A replay hit: the recorded claim extraction for one CIM block
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
out = llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks["p0002_b001"]["text"]), llm.P["llm"]["models"]["large"])
for c in out.claims: print(c.metric, c.value, c.unit, c.period, c.qualifier)

In [ ]:
# A replay miss is an error that names the missing key: replay never calls a provider
try:
    llm.structured(schemas.MgmtClaims, prompts.claims_messages("A sentence nobody recorded."), llm.P["llm"]["models"]["large"])
except RuntimeError as e:
    print("as expected:", e)

In [ ]:
# The cost ledger and the budget cap (part 3): what a live call of 1,000 tokens in and 300 out would cost
print("USD:", llm.usd(llm.P["llm"]["models"]["large"], 1000, 300), "| cap:", llm.CAP, "| spent so far:", llm.spent())

**Exercise 0.** Switch to record mode with your own key (`export LANTERN_LLM_MODE=record` and `OPENAI_API_KEY=...`), ask one new extraction, and confirm that a cache file and one ledger row appear. Then switch back to replay and run it again offline.